# Portion pipeline v4 — recipe-level EDA

Recipe-level resolution analysis for the **v4 no-portion retry** run
(`scratch/EDA/portion_feasibility_1000_v4_no_portion/`).

Questions answered:
- How many recipes have **all** ingredient lines with resolved `grams`?
- How many are **all but one** unresolved?
- How are sampled recipes distributed across RecipeNLG **source** and **link domain**?
- Does resolution rate vary by domain?

**Kernel cwd:** `Capstone/` or `scratch/EDA/`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def resolve_capstone_root() -> Path:
    cwd = Path.cwd()
    for candidate in (cwd, cwd.parent, cwd.parent.parent):
        if (candidate / "scripts" / "db.py").is_file():
            return candidate
    raise FileNotFoundError("Run with kernel cwd = Capstone/ or scratch/EDA/")


ROOT = resolve_capstone_root()
V4_DIR = ROOT / "scratch" / "EDA" / "portion_feasibility_1000_v4_no_portion"
RECIPE_CSV = ROOT / "Data" / "recipes" / "RecipeNLG.csv"
CACHE_PATH = V4_DIR / "recipe_level_eda.parquet"

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)

print(f"Capstone root: {ROOT}")
print(f"V4 run dir:    {V4_DIR}")

## Load v4 pipeline output + build recipe-level frame

A line is **gram-resolved** when `grams` is non-null (includes `compound_skipped` → 0g).

In [ ]:
REBUILD = not CACHE_PATH.is_file()

report = json.loads((V4_DIR / "feasibility_report.json").read_text())
lines = pd.read_parquet(V4_DIR / "pipeline_matches.parquet")
amount = pd.read_parquet(V4_DIR / "amount_classification.parquet")

lines["has_grams"] = lines["grams"].notna()
lines["has_fdc"] = lines["llm_fdc_id"].notna()
lines["resolved_both"] = lines["has_fdc"] & lines["has_grams"]

print(f"Lines: {len(lines):,} | Recipes in report: {report['n_recipes']:,}")
print(f"Line-level fdc+grams: {lines['resolved_both'].mean():.1%}")

In [ ]:
def link_domain(link) -> str:
    if link is None or (isinstance(link, float) and pd.isna(link)):
        return "(empty)"
    s = str(link).strip()
    if not s:
        return "(empty)"
    if not s.startswith(("http://", "https://")):
        s = "http://" + s
    host = urlparse(s).netloc.lower().replace("www.", "")
    return host or "(empty)"


def load_recipe_metadata(recipe_ids: list[int]) -> pd.DataFrame:
    id_set = set(int(i) for i in recipe_ids)
    parts: list[pd.DataFrame] = []
    for chunk in pd.read_csv(RECIPE_CSV, chunksize=200_000):
        id_col = chunk.columns[0]
        sel = chunk[chunk[id_col].astype(int).isin(id_set)]
        if len(sel):
            parts.append(sel)
    if not parts:
        return pd.DataFrame()
    meta = pd.concat(parts, ignore_index=True)
    id_col = meta.columns[0]
    meta = meta.rename(columns={id_col: "recipe_id"})
    meta["recipe_id"] = meta["recipe_id"].astype(int)
    meta["link_domain"] = meta["link"].map(link_domain)
    return meta.sort_values("recipe_id").reset_index(drop=True)


if REBUILD:
    recipe_ids = report["sampled_recipe_ids"]
    meta = load_recipe_metadata(recipe_ids)

    rec = (
        lines.groupby("recipe_id", as_index=False)
        .agg(
            n_lines=("ingredient_idx", "count"),
            n_resolved=("has_grams", "sum"),
            n_fdc=("has_fdc", "sum"),
            n_both=("resolved_both", "sum"),
            n_no_portion=("grams_status", lambda s: (s == "no_portion").sum()),
            n_missing_fdc=("grams_status", lambda s: (s == "missing_fdc").sum()),
        )
    )
    rec["n_unresolved"] = rec["n_lines"] - rec["n_resolved"]
    rec["all_resolved"] = rec["n_unresolved"] == 0
    rec["all_but_one"] = rec["n_unresolved"] == 1
    rec["gram_resolve_rate"] = (rec["n_resolved"] / rec["n_lines"]).round(4)
    rec["fdc_and_gram_rate"] = (rec["n_both"] / rec["n_lines"]).round(4)

    recipe_df = rec.merge(
        meta[["recipe_id", "title", "link", "source", "link_domain"]],
        on="recipe_id",
        how="left",
    )
    recipe_df.to_parquet(CACHE_PATH, index=False)
    print(f"Built recipe_df ({len(recipe_df):,} recipes) → {CACHE_PATH}")
else:
    recipe_df = pd.read_parquet(CACHE_PATH)
    print(f"Loaded cached recipe_df ({len(recipe_df):,} recipes) → {CACHE_PATH}")

recipe_df.head(3)

## Headline: recipe-level gram resolution

| Bucket | Definition |
|--------|------------|
| **All resolved** | `n_unresolved == 0` — every line has `grams` |
| **All but one** | exactly one line missing `grams` |
| **≤1 unresolved** | all resolved or all but one |

In [ ]:
n_recipes = len(recipe_df)
all_resolved = int(recipe_df["all_resolved"].sum())
all_but_one = int(recipe_df["all_but_one"].sum())
at_most_one = int((recipe_df["n_unresolved"] <= 1).sum())

headline = pd.DataFrame([
    {"bucket": "all lines gram-resolved", "n_recipes": all_resolved, "pct": round(all_resolved / n_recipes, 4)},
    {"bucket": "all but 1 unresolved", "n_recipes": all_but_one, "pct": round(all_but_one / n_recipes, 4)},
    {"bucket": "≤1 unresolved (union)", "n_recipes": at_most_one, "pct": round(at_most_one / n_recipes, 4)},
])
display(headline)

print(f"\nMedian unresolved lines per recipe: {recipe_df['n_unresolved'].median():.0f}")
print(f"Mean unresolved lines per recipe:   {recipe_df['n_unresolved'].mean():.2f}")

In [ ]:
dist = (
    recipe_df["n_unresolved"]
    .value_counts()
    .sort_index()
    .rename_axis("unresolved_lines")
    .reset_index(name="n_recipes")
)
dist["pct_recipes"] = (dist["n_recipes"] / n_recipes).round(4)
dist["cum_pct"] = dist["pct_recipes"].cumsum().round(4)
display(dist.head(12))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(dist["unresolved_lines"].astype(str), dist["n_recipes"], color="steelblue", edgecolor="white")
ax.set_xlabel("Unresolved ingredient lines per recipe")
ax.set_ylabel("Number of recipes")
ax.set_title("v4 run: distribution of unresolved lines per recipe (n=1,000)")
plt.tight_layout()
plt.show()

## RecipeNLG metadata: `source` and link domain

RecipeNLG columns used:
- **`source`** — dataset provenance label (e.g. `Gathered`, `Recipes1M`)
- **`link`** — original recipe URL; we derive **`link_domain`** (e.g. `cookbooks.com`, `food.com`)

In [ ]:
print("=== RecipeNLG `source` column (sampled 1,000 recipes) ===")
src = recipe_df["source"].value_counts().rename_axis("source").reset_index(name="n_recipes")
src["pct"] = (src["n_recipes"] / n_recipes).round(4)
display(src)

fig, ax = plt.subplots(figsize=(5, 3))
ax.pie(src["n_recipes"], labels=src["source"], autopct="%1.1f%%", startangle=90)
ax.set_title("NLG source label")
plt.tight_layout()
plt.show()

In [ ]:
print("=== Link domain breakdown (top 20) ===")
dom = recipe_df["link_domain"].value_counts().head(20).rename_axis("link_domain").reset_index(name="n_recipes")
dom["pct"] = (dom["n_recipes"] / n_recipes).round(4)
display(dom)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(dom["link_domain"][::-1], dom["n_recipes"][::-1], color="coral", edgecolor="white")
ax.set_xlabel("Recipes in sample")
ax.set_title("Top link domains in 1,000-recipe feasibility sample")
plt.tight_layout()
plt.show()

## Resolution rate by domain

Compare line-level and recipe-level success across link domains (domains with ≥5 recipes in sample).

In [ ]:
lines_meta = lines.merge(
    recipe_df[["recipe_id", "source", "link_domain", "title"]],
    on="recipe_id",
    how="left",
)

def domain_stats(group_col: str, min_recipes: int = 5) -> pd.DataFrame:
    g = lines_meta.groupby(group_col)
    line_stats = g.agg(
        n_lines=("ingredient_idx", "count"),
        gram_rate=("has_grams", "mean"),
        fdc_and_gram_rate=("resolved_both", "mean"),
    )
    rec_stats = recipe_df.groupby(group_col).agg(
        n_recipes=("recipe_id", "count"),
        all_resolved_rate=("all_resolved", "mean"),
        all_but_one_rate=("all_but_one", "mean"),
        mean_unresolved=("n_unresolved", "mean"),
    )
    out = line_stats.join(rec_stats)
    out = out[out["n_recipes"] >= min_recipes].sort_values("gram_rate", ascending=False)
    for c in ["gram_rate", "fdc_and_gram_rate", "all_resolved_rate", "all_but_one_rate"]:
        out[c] = out[c].round(4)
    out["mean_unresolved"] = out["mean_unresolved"].round(2)
    return out.reset_index()

print("By link_domain (≥5 recipes):")
display(domain_stats("link_domain"))

print("\nBy NLG source label:")
display(domain_stats("source", min_recipes=1))

## Spot checks: fully resolved recipes and near-misses

Sample recipes that are **100% resolved** vs **all but one**.

In [ ]:
cols = ["recipe_id", "title", "link_domain", "source", "n_lines", "n_unresolved", "gram_resolve_rate"]

print("Fully resolved recipes (first 15):")
display(recipe_df.loc[recipe_df["all_resolved"], cols].head(15))

print("\nAll-but-one recipes (first 15) — with the unresolved line:")
near = recipe_df.loc[recipe_df["all_but_one"], cols].head(15)
display(near)

if len(near):
    sample_id = int(near.iloc[0]["recipe_id"])
    unresolved = lines_meta[(lines_meta["recipe_id"] == sample_id) & (~lines_meta["has_grams"])]
    show_cols = ["ingredient", "amount_kind_final", "grams_status", "llm_fdc_id", "llm_description"]
    print(f"\nUnresolved line(s) for recipe {sample_id}:")
    display(unresolved[show_cols])

## Export

Uncomment to write a CSV for downstream use.

In [ ]:
# out_csv = V4_DIR / "recipe_level_v4_summary.csv"
# recipe_df.to_csv(out_csv, index=False)
# print(f"Wrote {out_csv}")